In [ ]:
# ================================================
# Author: Aida Katkauskaitė
# LSP Number: 2516024
# Variant: 1
# Task 1: Image Segmentation
# Classes: Dog (1), Car (2), Person (3)
# Version: 1.0
# ================================================

In [1]:
# For importing images from fiftyone
#import fiftyone as fo
#import fiftyone.zoo as foz

# Math, image preparation, file handling and random number generation libraries
# time for tracking epoch running times
import os
import shutil
import random
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import time

# Libraries for model training
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF

# Libraries for model evaluation
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

In [ ]:
# On local python notebook I used this code to download image datasets.
# For each group downloaded 2000 images and manually copied to other folders.

dataset = foz.load_zoo_dataset(
        "open-images-v7",
        split="train",
        label_types=["segmentations"],
        classes="Dog",
        max_samples=2000
    )

In [ ]:
# From all of the masks which were downloaded into folders 0 and 1 filtered the
# masks which were for images in the folder with images (by image name and mask name beginning)
# added these masks into folder B. And done this for all classes by class codes
# in the mask namse ("m0bt9lr", "m0k4j", "m01g317")

src = r"C:\Users\User\fiftyone\Dog\train\labels\masks\1"
dst = r"C:\Users\User\fiftyone\Dog\train\labels\masks\B"
img_dir = r"C:\Users\User\fiftyone\Dog\train\data"

image_ids = set(os.listdir(img_dir))

for filename in os.listdir(src):
    image_id = filename.split("_")[0] + ".jpg"
    if "m0bt9lr" in filename and image_id in image_ids:
        shutil.copy(os.path.join(src, filename), dst)

In [ ]:
# Resized images and masks to save time on training
src_imgs = r"C:\Users\User\fiftyone\train\Data"
src_masks = r"C:\Users\User\fiftyone\train\masks"
dst_imgs = r"C:\Users\User\fiftyone\train\Data_resized"
dst_masks = r"C:\Users\User\fiftyone\train\Masks_resized"

SIZE = (256, 256)

for f in os.listdir(src_imgs):
    Image.open(os.path.join(src_imgs, f)).resize(SIZE).save(os.path.join(dst_imgs, f))

for f in os.listdir(src_masks):
    Image.open(os.path.join(src_masks, f)).resize(SIZE, Image.NEAREST).save(os.path.join(dst_masks, f))

In [ ]:
# Combining masks together for each image, each pixel gets mapped to background, dog, car or person
src_masks = r"C:\Users\User\fiftyone\train\Masks_resized"
dst_masks = r"C:\Users\User\fiftyone\train\training_masks"
img_dir = r"C:\Users\User\fiftyone\train\Data_resized"

# Class codes from open images assigned values
CLASS_CODES = {"m0bt9lr": 1, "m0k4j": 2, "m01g317": 3}

for image_file in os.listdir(img_dir):
    image_id = os.path.splitext(image_file)[0]
    #First assigns class background to all images
    combined_mask = np.zeros((256, 256), dtype=np.uint8)

    for mask_file in os.listdir(src_masks):
        if not mask_file.startswith(image_id):
            continue
    # Checks which class mask belongs to based on filename code
        for code, class_idx in CLASS_CODES.items():
            if code in mask_file:
                mask = np.array(Image.open(os.path.join(src_masks, mask_file)))
    # For all non-zero pixels in the mask adds class index
                                combined_mask[mask > 0] = class_idx

    Image.fromarray(combined_mask).save(os.path.join(dst_masks, f"{image_id}.png"))

In [2]:
#After preparation uploaded data to google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
#Copy images to disc
shutil.copytree("/content/drive/MyDrive/segmentation/Data_resized", "/content/Data_resized")
shutil.copytree("/content/drive/MyDrive/segmentation/training_masks", "/content/training_masks")

'/content/training_masks'

In [4]:
# ------------Creating dataset class which is needed for loading
#and preparing the data for model

class SegmentationDataset(Dataset):
# Sorting images and masks to get matching pairs
    def __init__(self, img_dir, mask_dir):
        self.imgs = sorted(os.listdir(img_dir))
        self.masks = sorted(os.listdir(mask_dir))
        self.img_dir = img_dir
        self.mask_dir = mask_dir

# Function which defines dataset length needed for PyTorch
    def __len__(self):
        return len(self.imgs)

# Loading image and mask pairs
# images are converted to rgb so that all have 3 color channels as it is needed for UNet
    def __getitem__(self, i):
        img = Image.open(os.path.join(self.img_dir,  self.imgs[i])).convert("RGB")
        mask = Image.open(os.path.join(self.mask_dir, self.masks[i]))

#------------ Image augmentation
# Apllying random transformations

#1 flipping we assign random number based on which image is fliped.
#adding 50% chance for this, could be changed.
#applying for images and masks, for pixels to still match
        if random.random() > 0.5:
            img = TF.hflip(img)
            mask = TF.hflip(mask)

#2 changing brightness for images randomly, again 50% chance, brighness value
#also randomly assigned between 0.7 and 1.3
        if random.random() > 0.5:
            img = TF.adjust_brightness(img, random.uniform(0.7, 1.3))

#3 rotating images and their masks by random angle from -30 to +30
# with 50% chance of rotation
        if random.random() > 0.5:
            angle = random.randint(-30, 30)
            img = TF.rotate(img, angle)
            mask = TF.rotate(mask, angle)

#----------------Converting images and masks to PyTorch Tensors
        img = T.ToTensor()(img)

# torch.long is used so class values could be afterwards
        mask = torch.tensor(np.array(mask), dtype=torch.long)
        return img, mask

In [5]:
# Definition of convolution function. Two layers, 3 by 3.
# Activation function ReLU
def conv_block(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
        nn.ReLU(),
    )


# Using UNet neural network design
# Encoder shrinks the image to find patterns
# Decoder expands back to full size to label each pixel
# Skip connections pass detail from encoder directly to decoder
class UNet(nn.Module):
    def __init__(self):
# From Pytorch
        super().__init__()
# First encoder 3 RGB channels and 64 feature maps
# Second encoder 64 feture maps converted to 128
# pool splits imahe in half each step
        self.enc1 = conv_block(3, 64)
        self.enc2 = conv_block(64, 128)
        self.pool = nn.MaxPool2d(2)

        self.mid = conv_block(128, 256)

# Decoders doubles image size unsamples in levels 256->128->64
        self.up1 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec1 = conv_block(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = conv_block(128, 64)

# Converting features to class scores for 4 classes
        self.out = nn.Conv2d(64, 4, kernel_size=1)


# Go through encoders and decoders
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        m = self.mid(self.pool(e2))
        d1 = self.dec1(torch.cat([self.up1(m), e2], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d1), e1], dim=1))
        return self.out(d2)

In [6]:
#------------------Training setup
# device chacks if GPU is available
# First tried on local computer with only CPU it was to slow
# Trained further in colob on GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Creating dataset and data loader
# loads and shuffles 16 images at a time
dataset = SegmentationDataset("/content/Data_resized", "/content/training_masks")
loader = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=2)
# Dataset contains 5907 images
print(f"Dataset size: {len(dataset)}")

# starts the model and move it to GPU or CPU depending which available
model = UNet().to(device)

# Locally checked % of pixels, background class was most abundant, and person class was misrepresented
# Background: 316557325 (81.8%)
#Dog: 28466698 (7.4%)
#Car: 30080264 (7.8%)
#Person: 12016865 (3.1%)


# first 40 epochs of the model were trained with cross entropy loss function
#with weights and learing rate as follows
#loss_fn   = nn.CrossEntropyLoss(weight=weights)
#weights   = torch.tensor([0.25, 1.0, 1.0, 1.0]).to(device)
#optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
# results on unseen data were

#for 10 more epochs learning rate was changed to lr=0.00001
# background was reduces and person weight increased
#weights = torch.tensor([0.2, 1.0, 1.0, 2.0]).to(device)

# Model result for 100 unseen images for 50 epochs
#Accuracy:0.693
#background:precision=0.948;recall=0.684;F1=0.795
#dog:precision=0.589;recall=0.762;F1=0.665
#car:precision=0.268;recall=0.798;F1=0.401
#person:precision=0.059;recall=0.247;F1=0.095

#Last weights
weights   = torch.tensor([1, 1.0, 1.0, 2.0]).to(device)

# After this, loss function was changed to Focal loss as there is a lot of background
# and smaller group for persons
# with weights 0.25, 1.0, 1.0, 4.0 it got worse after 20 epochs

class FocalLoss(nn.Module):
    def __init__(self, gamma=2, weight=None):
        super().__init__()
        self.gamma = gamma
        self.weight = weight

    def forward(self, pred, target):
        ce_loss = nn.functional.cross_entropy(pred, target, weight=self.weight, reduction='none')
        pt = torch.exp(-ce_loss)
        return ((1 - pt) ** self.gamma * ce_loss).mean()

loss_fn = FocalLoss(gamma=2, weight=weights)

optimizer = torch.optim.Adam(model.parameters(), lr=0.00001)

Dataset size: 5907


In [ ]:
# First model ran for 40 epochs, than 10 more, than loss function was changed and 20 more were ran
# model saved after ech epoch so it was possible to train further on started model
for epoch in range(20):
    start = time.time()
#starting training mode
    model.train()
    total_loss = 0

    for imgs, masks in loader:
# Move data to GPU/CPU depending on availability
        imgs, masks = imgs.to(device), masks.to(device)
# Zero gradients from previous batch
        optimizer.zero_grad()
# Model predictions
        loss = loss_fn(model(imgs), masks)
# Calculating how much each weight contributed to the error
        loss.backward()
# Update weights based on gradients
        optimizer.step()
        total_loss += loss.item()
# Just checking time and printing out
    elapsed = time.time() - start
    print(f"Epoch {epoch+1}/20  loss: {total_loss/len(loader):.4f}  time: {elapsed:.0f}s")
# Saving after every epoch
    torch.save(model.state_dict(), f"/content/drive/MyDrive/segmentation/unet_epoch_{epoch+1}.pth")

In [7]:
#Loading test images and masks
shutil.copytree("/content/drive/MyDrive/segmentation/data_resized", "/content/test_images")
shutil.copytree("/content/drive/MyDrive/segmentation/testing_masks", "/content/test_masks")

'/content/test_masks'

In [8]:
# To evaluate the model 100 test images were used
# Note: As after testing I did some adjustments and retraining on training data
# this was more of validation and not testing, should be tested again with new test data

#First model after 50 epochs
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = UNet().to(device)
model.load_state_dict(torch.load("/content/drive/MyDrive/segmentation/unet_epoch_50.pth"))

# Switching to model evaluation
model.eval()

test_img_dir = "/content/test_images"
test_mask_dir = "/content/test_masks"

all_preds = []
all_true = []

# torch.no_grad() skips computing gradients for faster evaluation
with torch.no_grad():
    for img_file in os.listdir(test_img_dir):
        image_id = os.path.splitext(img_file)[0]
        mask_file = f"{image_id}.png"

        if not os.path.exists(os.path.join(test_mask_dir, mask_file)):
            continue
# Converting images to tensor format and make 3 RGB channels
        img = T.ToTensor()(Image.open(os.path.join(test_img_dir, img_file)).convert("RGB"))
# Loading ground truth mask
        mask = np.array(Image.open(os.path.join(test_mask_dir, mask_file)))
# Adding bach dimension, move to defined device, picking class with highest score, removing bach
        pred = model(img.unsqueeze(0).to(device)).argmax(dim=1).squeeze(0).cpu().numpy()
# Flattening 2D pixel arrays to 1D
        all_preds.append(pred.flatten())
        all_true.append(mask.flatten())

# Concatenating to single arrays
y_pred = np.concatenate(all_preds)
y_true = np.concatenate(all_true)

# Computing metrics for each class
classes = ["background", "dog", "car", "person"]
acc = accuracy_score(y_true, y_pred)
p, r, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=[0,1,2,3], average=None, zero_division=0
)

# precision - % of correct predictions
# recall - % of actual class pixels found
# F1 - combines precision and recall
#F1 = 2 × (precision × recall) / (precision + recall)

print(f"Overall accuracy: {acc:.3f}\n")
for i, name in enumerate(classes):
    print(f"{name}: precision={p[i]:.3f}  recall={r[i]:.3f}  F1={f1[i]:.3f}")

Overall accuracy: 0.693

background: precision=0.948  recall=0.684  F1=0.795
dog: precision=0.589  recall=0.762  F1=0.665
car: precision=0.268  recall=0.798  F1=0.401
person: precision=0.059  recall=0.247  F1=0.095


In [9]:
# To evaluate the model 100 test images were used
# Note: As after testing I did some adjustments and retraining on training data
# this was more of validation and not testing, should be tested again with new test data

#Second try/ first try with focal loss function and different weights - got worse
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = UNet().to(device)
model.load_state_dict(torch.load("/content/drive/MyDrive/segmentation/unet_epoch_70.pth"))

# Switching to model evaluation
model.eval()

test_img_dir = "/content/test_images"
test_mask_dir = "/content/test_masks"

all_preds = []
all_true = []

# torch.no_grad() skips computing gradients for faster evaluation
with torch.no_grad():
    for img_file in os.listdir(test_img_dir):
        image_id = os.path.splitext(img_file)[0]
        mask_file = f"{image_id}.png"

        if not os.path.exists(os.path.join(test_mask_dir, mask_file)):
            continue
# Converting images to tensor format and make 3 RGB channels
        img = T.ToTensor()(Image.open(os.path.join(test_img_dir, img_file)).convert("RGB"))
# Loading ground truth mask
        mask = np.array(Image.open(os.path.join(test_mask_dir, mask_file)))
# Adding bach dimension, move to defined device, picking class with highest score, removing bach
        pred = model(img.unsqueeze(0).to(device)).argmax(dim=1).squeeze(0).cpu().numpy()
# Flattening 2D pixel arrays to 1D
        all_preds.append(pred.flatten())
        all_true.append(mask.flatten())

# Concatenating to single arrays
y_pred = np.concatenate(all_preds)
y_true = np.concatenate(all_true)

# Computing metrics for each class
classes = ["background", "dog", "car", "person"]
acc = accuracy_score(y_true, y_pred)
p, r, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=[0,1,2,3], average=None, zero_division=0
)

# precision - % of correct predictions
# recall - % of actual class pixels found
# F1 - combines precision and recall
#F1 = 2 × (precision × recall) / (precision + recall)

print(f"Overall accuracy: {acc:.3f}\n")
for i, name in enumerate(classes):
    print(f"{name}: precision={p[i]:.3f}  recall={r[i]:.3f}  F1={f1[i]:.3f}")

Overall accuracy: 0.417

background: precision=0.991  recall=0.343  F1=0.510
dog: precision=0.478  recall=0.763  F1=0.588
car: precision=0.237  recall=0.672  F1=0.350
person: precision=0.033  recall=0.730  F1=0.063


In [11]:
# To evaluate the model 100 test images were used
# Note: As after testing I did some adjustments and retraining on training data
# this was more of validation and not testing, should be tested again with new test data

#Third try/ second try with focal loss function and the weights defined here
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = UNet().to(device)
model.load_state_dict(torch.load("/content/drive/MyDrive/segmentation/unet_epoch_70.pth"))

# Switching to model evaluation
model.eval()

test_img_dir = "/content/test_images"
test_mask_dir = "/content/test_masks"

all_preds = []
all_true = []

# torch.no_grad() skips computing gradients for faster evaluation
with torch.no_grad():
    for img_file in os.listdir(test_img_dir):
        image_id = os.path.splitext(img_file)[0]
        mask_file = f"{image_id}.png"

        if not os.path.exists(os.path.join(test_mask_dir, mask_file)):
            continue
# Converting images to tensor format and make 3 RGB channels
        img = T.ToTensor()(Image.open(os.path.join(test_img_dir, img_file)).convert("RGB"))
# Loading ground truth mask
        mask = np.array(Image.open(os.path.join(test_mask_dir, mask_file)))
# Adding bach dimension, move to defined device, picking class with highest score, removing bach
        pred = model(img.unsqueeze(0).to(device)).argmax(dim=1).squeeze(0).cpu().numpy()
# Flattening 2D pixel arrays to 1D
        all_preds.append(pred.flatten())
        all_true.append(mask.flatten())

# Concatenating to single arrays
y_pred = np.concatenate(all_preds)
y_true = np.concatenate(all_true)

# Computing metrics for each class
classes = ["background", "dog", "car", "person"]
acc = accuracy_score(y_true, y_pred)
p, r, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=[0,1,2,3], average=None, zero_division=0
)

# precision - % of correct predictions
# recall - % of actual class pixels found
# F1 - combines precision and recall
#F1 = 2 × (precision × recall) / (precision + recall)

print(f"Overall accuracy: {acc:.3f}\n")
for i, name in enumerate(classes):
    print(f"{name}: precision={p[i]:.3f}  recall={r[i]:.3f}  F1={f1[i]:.3f}")

Overall accuracy: 0.826

background: precision=0.885  recall=0.906  F1=0.895
dog: precision=0.725  recall=0.592  F1=0.652
car: precision=0.386  recall=0.445  F1=0.413
person: precision=0.048  recall=0.013  F1=0.020


In [ ]:
#In all three runs person class is not identified correctly most of the time
#Although in the last version the overall result was the best, model still needs improvements